# Turtle Rules (SMA20/SMA55) on SPY
## Strategy Brief
The Turtle Rules strategy utilizes two simple moving averages (SMA) to generate trading signals on SPY. Specifically, it uses a short-term SMA (20 days) and a long-term SMA (55 days). When the short-term SMA crosses above the long-term SMA, it signals a buy, predicting an upward trend. Conversely, a cross below signals a sell, predicting a downward trend. This trend-following strategy aims to capitalize on sustained price movements. Results are typically evaluated based on risk-adjusted returns and compared against a buy-and-hold benchmark.
## References
- https://www.mql5.com/en/articles/23155

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
Define the trading parameters and constants used throughout the strategy.

In [ ]:
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
SHORT_SMA = 20
LONG_SMA = 55
TICKER = 'SPY'

## PHASE 2 - Data Exploration
Download historical data for SPY, compute the SMA indicators, and visualize them overlaid on the price data.

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Compute SMAs
data['SMA20'] = data['Close'].rolling(window=SHORT_SMA).mean()
data['SMA55'] = data['Close'].rolling(window=LONG_SMA).mean()

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='Close Price')
plt.plot(data['SMA20'], label='SMA20')
plt.plot(data['SMA55'], label='SMA55')
plt.title('SPY Price with SMA20 and SMA55')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
Develop the trading signal based on SMA crossovers and define the entry and exit logic for positions.

In [ ]:
# Generate signals
data['Signal'] = 0
# Buy signal
data.loc[data['SMA20'] > data['SMA55'], 'Signal'] = 1
# Sell signal
data.loc[data['SMA20'] < data['SMA55'], 'Signal'] = -1

# Entry/Exit logic
data['Position'] = data['Signal'].shift(1)

## PHASE 4 - Coding & Backtesting
Calculate daily returns based on positions and plot the resulting equity curve.

In [ ]:
# Calculate daily returns
data['Market Return'] = data['Close'].pct_change()
data['Strategy Return'] = data['Market Return'] * data['Position']

# Compute equity curve
data['Equity Curve'] = (1 + data['Strategy Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity Curve'], label='Equity Curve')
plt.title('Equity Curve of the Turtle Rules Strategy')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
Evaluate the strategy's performance using key metrics and compare it to a buy-and-hold strategy.

In [ ]:
import numpy as np

# Calculate performance metrics
days = (data.index[-1] - data.index[0]).days
cagr = (data['Equity Curve'].iloc[-1])**(365.0/days) - 1
sharpe_ratio = np.mean(data['Strategy Return']) / np.std(data['Strategy Return']) * np.sqrt(252)

# Sortino Ratio
negative_volatility = np.std(data.loc[data['Strategy Return'] < 0, 'Strategy Return'])
sortino_ratio = np.mean(data['Strategy Return']) / negative_volatility * np.sqrt(252)

# Calmar Ratio
max_drawdown = (data['Equity Curve'].cummax() - data['Equity Curve']).max()
calmar_ratio = cagr / max_drawdown

# Buy-and-hold comparison
buy_and_hold_return = (data['Close'].iloc[-1] / data['Close'].iloc[0]) - 1

# Print results
print(f"CAGR: {cagr:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Sortino Ratio: {sortino_ratio:.2f}")
print(f"Calmar Ratio: {calmar_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Buy-and-Hold Return: {buy_and_hold_return:.2%}")

## PHASE 6 - Deploy & Monitor
Create a function to download recent data, compute the current signal, and print the trading position.

In [ ]:
def get_current_signal():
    recent_data = yf.download(TICKER, period='60d')
    recent_data['SMA20'] = recent_data['Close'].rolling(window=SHORT_SMA).mean()
    recent_data['SMA55'] = recent_data['Close'].rolling(window=LONG_SMA).mean()
    
    if recent_data['SMA20'].iloc[-1] > recent_data['SMA55'].iloc[-1]:
        print("Current Signal: Buy")
    elif recent_data['SMA20'].iloc[-1] < recent_data['SMA55'].iloc[-1]:
        print("Current Signal: Sell")
    else:
        print("Current Signal: Hold")

get_current_signal()